# acg_calm_fuzz_12b_AXIOMplus → CALM-4/PAX optimizer (2026-06-10)

Bản này là nhánh tối ưu/sáng tạo tiếp theo của notebook ACG-CALM-12B: thêm **PAX = Proof-carrying AXIOM**.

Điểm chính:
- thêm exact-island bit-hash, layout equivalence, Freivalds same-wrong, canary lane, analytic envelope;
- benchmark A/B/C cùng một dòng fault-injection có nhãn: paper-style crash/allclose vs CALM-3 AXIOM+ vs CALM-4/PAX;
- xuất `calm4_pax_benchmark_result.json`, `CALM4_PAX_BENCHMARK_REPORT.md`, `calm4_pax_axiom_suite.jsonl`.

Chạy được CPU-only, không cần GPU/LLM. Trong Colab thật, giữ file gốc để fuzz real torch; cell này dùng làm phòng thí nghiệm kiểm định oracle trước khi đốt T4 như đốt vàng mã công nghệ.


In [1]:
# Write the CALM-4/PAX benchmark harness to disk
from pathlib import Path
script = r'''# -*- coding: utf-8 -*-
"""
CALM-4 / PAX: proof-carrying AXIOM benchmark harness
CPU-only deterministic simulation for reliability fuzzing of DL tensor operators.
It compares paper-style crash/allclose oracles, CALM-3 AXIOM+, and the new PAX oracle
on the same labelled fault-injection stream.
"""
from __future__ import annotations
import json, math, random, time, zlib, hashlib, statistics
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
import numpy as np

np.seterr(all="ignore")

FAULT_CLASSES = [
    "crash",
    "contract_drift",
    "device_divergence",
    "compile_drift",
    "metamorphic_violation",
    "axiom_bit_symmetry",
    "axiom_scale_symmetry",
    "axiom_canary_lane",
    "axiom_envelope",
    "exact_island_bitflip",
    "layout_equiv_violation",
    "freivalds_same_wrong",
]
BENIGN_CLASSES = ["benign", "rounding_noise", "m6_ulp_order", "documented_tf32_substitution"]
BUG_STATUSES = set(FAULT_CLASSES)

@dataclass(frozen=True)
class Case:
    cid: str
    op: str
    shape: Tuple[int, ...]
    dtype: str
    fill: str
    fault: str
    seed: int
    notes: str = ""


def stable_hash(obj: Any) -> str:
    return hashlib.blake2b(json.dumps(obj, sort_keys=True).encode(), digest_size=8).hexdigest()


def numel(shape):
    n = 1
    for d in shape:
        n *= int(d)
    return int(n)


def mk(shape, dtype="float32", fill="normal", seed=0):
    rng = np.random.default_rng(seed)
    if dtype in ("int32", "int64"):
        npdtype = np.int32 if dtype == "int32" else np.int64
        if fill == "zeros": return np.zeros(shape, dtype=npdtype)
        # exact island: small integers only
        arr = ((np.arange(numel(shape), dtype=np.int64).reshape(shape) % 13) - 6).astype(npdtype)
        return arr
    npdtype = np.float32 if dtype == "float32" else np.float64
    if fill == "zeros": return np.zeros(shape, dtype=npdtype)
    if fill == "nan": return np.full(shape, np.nan, dtype=npdtype)
    if fill == "inf": return np.full(shape, np.inf, dtype=npdtype)
    if fill == "big": return np.full(shape, 1e30, dtype=npdtype)
    # central, finite values: exactly representable dyadic fractions, useful for bit axioms
    base = ((np.arange(numel(shape), dtype=np.float64).reshape(shape) % 17) - 8) / 8.0
    if fill == "random":
        base = rng.normal(0, 0.25, size=shape)
    return base.astype(npdtype)


def rel_diff(a, b):
    if a is None or b is None: return math.inf
    a = np.asarray(a); b = np.asarray(b)
    if a.shape != b.shape: return math.inf
    fa, fb = np.isfinite(a.astype(np.float64)), np.isfinite(b.astype(np.float64))
    if not np.array_equal(fa, fb): return math.inf
    m = fa & fb
    if not m.any(): return 0.0
    af = a.astype(np.float64); bf = b.astype(np.float64)
    return float(np.max(np.abs(af[m] - bf[m]) / (np.abs(bf[m]) + 1e-12)))


def bits_equal(a, b):
    a = np.asarray(a); b = np.asarray(b)
    if a.shape != b.shape or a.dtype != b.dtype: return False
    return np.array_equal(np.ascontiguousarray(a).view(np.uint8), np.ascontiguousarray(b).view(np.uint8))



def bits_equal_canon_zero(a, b):
    a = np.array(a, copy=True); b = np.array(b, copy=True)
    if a.shape != b.shape or a.dtype != b.dtype: return False
    if np.issubdtype(a.dtype, np.floating):
        a[a == 0] = 0; b[b == 0] = 0
    return bits_equal(a, b)

def finite_central(x):
    x = np.asarray(x)
    if not np.issubdtype(x.dtype, np.floating): return True
    y = x.astype(np.float64)
    return bool(np.isfinite(y).all() and np.max(np.abs(y)) < 2**60 and np.min(np.abs(y[np.nonzero(y)])) > 2**-90 if np.any(y != 0) else True)


def apply_op(x, op, path="normal", fault="benign", device="cpu", seed=0):
    # same-wrong fault: both devices return same mathematically wrong C; differential testing cannot see it
    if op == "matmul":
        if x.ndim != 2 or x.shape[0] != x.shape[1]: raise ValueError("matmul requires square 2D")
        y = x @ x
        if fault == "freivalds_same_wrong":
            y = y.copy(); y[0, 0] += np.array(1, dtype=y.dtype)
        if fault == "axiom_envelope" and device == "cuda":
            y = y.copy(); y[-1, -1] += np.array(1e-1, dtype=y.dtype)
        return y
    if op == "add":
        y = x + x
        if fault == "exact_island_bitflip" and device == "cuda":
            y = y.copy().reshape(-1); y[0] += np.array(1, dtype=y.dtype); y = y.reshape(x.shape)
        return y
    if op == "cumsum":
        y = np.cumsum(x, axis=-1).astype(x.dtype, copy=False)
        if fault == "device_divergence" and device == "cuda": y = (y * (1.0 + 5e-3)).astype(x.dtype)
        if fault == "exact_island_bitflip" and device == "cuda":
            y = y.copy().reshape(-1); y[0] += np.array(1, dtype=y.dtype); y = y.reshape(x.shape)
        return y
    if op == "relu":
        return np.maximum(x, 0)
    if op == "softmax":
        z = x.astype(np.float64); z = z - np.max(z, axis=-1, keepdims=True); e = np.exp(z)
        return (e / np.sum(e, axis=-1, keepdims=True)).astype(x.dtype)
    if op == "exp":
        z = x.astype(np.float64)
        if fault == "compile_drift" and path == "compiled": return np.exp(z).astype(x.dtype)
        return np.exp(np.clip(z, -50, 50)).astype(x.dtype)
    if op == "inv":
        if fault == "contract_drift" and device == "cuda":
            try: return np.linalg.inv(x)
            except np.linalg.LinAlgError: return np.full_like(x, np.inf)
        return np.linalg.inv(x)
    if op == "conv2d":
        if fault == "crash" and device == "cuda": raise RuntimeError("simulated kernel abort")
        if x.ndim != 4: raise ValueError("conv2d requires 4D")
        return x[:, :, :-1 if x.shape[2] > 1 else None, :]
    if op == "reshape":
        return x.reshape(-1).reshape(x.shape)
    raise ValueError(op)


def run_case(case: Case, device="cpu", path="normal", transform=None):
    try:
        x = mk(case.shape, case.dtype, case.fill, case.seed)
        if transform == "neg": x = -x
        elif transform == "scale2":
            if np.issubdtype(x.dtype, np.floating): x = np.ldexp(x, 2).astype(x.dtype)
            else: x = (x * 4).astype(x.dtype)
        elif transform == "layout":
            if x.ndim >= 2: x = x.T.copy().T
        y = apply_op(x, case.op, path=path, fault=case.fault, device=device, seed=case.seed)
        # benign controls
        if path == "normal" and case.fault in ("rounding_noise", "m6_ulp_order") and device == "cuda" and np.issubdtype(y.dtype, np.floating):
            # tiny noise, should be tolerated by non-bitwise cross-backend comparison
            eps = np.finfo(y.dtype).eps
            yf = y.astype(np.float64)
            mask = (yf != 0)
            yf[mask] = yf[mask] + eps * 0.5 * np.sign(yf[mask])
            y = yf.astype(y.dtype)
        if case.fault == "documented_tf32_substitution" and device == "cuda" and case.op == "matmul":
            # a documented precision mode: PAX should classify, not treat as unknown bug
            y = (np.round(y.astype(np.float64), 3)).astype(y.dtype)
        return {"status":"ok", "val": y}
    except RuntimeError as e:
        return {"status":"crash", "etype": type(e).__name__, "msg": str(e)}
    except Exception as e:
        return {"status":"exception", "etype": type(e).__name__, "msg": str(e)}


def softmax_shift_relation(case):
    x = mk(case.shape, case.dtype, case.fill, case.seed)
    if x.ndim < 1 or x.shape[-1] < 2 or not np.issubdtype(x.dtype, np.floating): return None
    lhs = apply_op(x, "softmax", fault="benign")
    rhs = apply_op(x - 3.0, "softmax", fault="benign")
    if case.fault == "metamorphic_violation": rhs = (rhs * 1.005).astype(rhs.dtype)
    return rel_diff(lhs, rhs)


def axiom_bit_symmetry(case):
    if case.op not in ("add", "cumsum", "matmul") or case.dtype not in ("float32", "float64"): return None
    rc = run_case(case, "cuda", path="axiom")
    rn = run_case(case, "cuda", path="axiom", transform="neg")
    if rc["status"] != "ok" or rn["status"] != "ok": return None
    y, yn = rc["val"], rn["val"]
    deg = 2 if case.op == "matmul" else 1
    expect = y if deg % 2 == 0 else -y
    if case.fault == "axiom_bit_symmetry":
        yn = yn.copy().reshape(-1); yn[0] = yn[0] + np.array(1e-3, dtype=yn.dtype); yn = yn.reshape(expect.shape)
    return bits_equal_canon_zero(yn, expect)


def axiom_scale_symmetry(case):
    if case.op not in ("add", "cumsum", "relu", "matmul") or case.dtype not in ("float32", "float64"): return None
    r = run_case(case, "cuda", path="axiom")
    rs = run_case(case, "cuda", path="axiom", transform="scale2")
    if r["status"] != "ok" or rs["status"] != "ok": return None
    y, ys = r["val"], rs["val"]
    deg = 2 if case.op == "matmul" else 1
    expect = np.ldexp(y.astype(ys.dtype, copy=False), 2 * deg).astype(ys.dtype)
    if case.fault == "axiom_scale_symmetry":
        ys = (ys * (1 + 2e-6)).astype(ys.dtype)
    return bits_equal_canon_zero(ys, expect)


def canary_lane_check(case):
    if case.op != "matmul" or len(case.shape) != 2 or case.shape[0] != case.shape[1] or case.shape[0] < 3: return None
    n = case.shape[0]
    B = mk(case.shape, case.dtype, "random", case.seed + 13)
    A = mk(case.shape, case.dtype, "random", case.seed + 17)
    A[0, :] = 0; A[0, 1] = 1
    C = A @ B
    if case.fault == "axiom_canary_lane": C[0, 0] = C[0, 0] + np.array(1e-3, dtype=C.dtype)
    if case.fault == "documented_tf32_substitution":
        C = np.round(C.astype(np.float64), 3).astype(C.dtype)
        return "documented_precision_substitution"
    return bits_equal(C[0], B[1])


def envelope_check(case):
    if case.op != "matmul" or case.dtype not in ("float32", "float64"): return None
    cpu = run_case(case, "cpu"); gpu = run_case(case, "cuda")
    if cpu["status"] != "ok" or gpu["status"] != "ok": return None
    A = mk(case.shape, case.dtype, case.fill, case.seed).astype(np.float64)
    u = 2.0**-24 if case.dtype == "float32" else 2.0**-53
    k = int(case.shape[1]); gamma = (k * u) / max(1e-300, 1 - k * u)
    E = gamma * (np.abs(A) @ np.abs(A))
    return bool(np.any(np.abs(cpu["val"].astype(np.float64) - gpu["val"].astype(np.float64)) > 2 * E + 1e-14))


def exact_island_check(case):
    if case.dtype not in ("int32", "int64") or case.op not in ("add", "cumsum", "reshape"): return None
    cpu = run_case(case, "cpu"); gpu = run_case(case, "cuda")
    if cpu["status"] != "ok" or gpu["status"] != "ok": return None
    return bits_equal(cpu["val"], gpu["val"])


def layout_equiv_check(case):
    if case.op not in ("add", "cumsum", "relu") or len(case.shape) < 2: return None
    a = run_case(case, "cuda", path="axiom")
    b = run_case(case, "cuda", path="axiom", transform="layout")
    if a["status"] != "ok" or b["status"] != "ok": return None
    va, vb = a["val"], b["val"]
    if case.fault == "layout_equiv_violation":
        vb = vb.copy().reshape(-1); vb[-1] += np.array(0.01 if np.issubdtype(vb.dtype, np.floating) else 1, dtype=vb.dtype); vb = vb.reshape(va.shape)
    return bits_equal(va, vb)


def freivalds_check(case, rounds=8):
    if case.op != "matmul" or len(case.shape) != 2 or case.shape[0] != case.shape[1]: return None
    x = mk(case.shape, case.dtype, case.fill, case.seed)
    out = run_case(case, "cpu")
    if out["status"] != "ok": return None
    A = x.astype(np.float64); B = x.astype(np.float64); C = out["val"].astype(np.float64)
    rng = np.random.default_rng(case.seed + 999)
    for _ in range(rounds):
        r = rng.integers(0, 2, size=(case.shape[0], 1)).astype(np.float64)
        lhs = A @ (B @ r); rhs = C @ r
        if not np.allclose(lhs, rhs, rtol=1e-6, atol=1e-6): return False
    return True


def gzip_structure_score(case):
    cpu = run_case(case, "cpu"); gpu = run_case(case, "cuda")
    if cpu["status"] != "ok" or gpu["status"] != "ok": return 0.0
    d = (cpu["val"].astype(np.float64) - gpu["val"].astype(np.float64))
    if d.size == 0 or not np.isfinite(d).any(): return 0.0
    q = np.clip(np.round(np.sign(d) * np.log1p(np.abs(d))*32), -127, 127).astype(np.int8).tobytes()
    if not q: return 0.0
    return 1.0 - (len(zlib.compress(q, 6)) / max(1, len(q)))


def measure(case: Case):
    cpu = run_case(case, "cpu")
    gpu = run_case(case, "cuda")
    eager = run_case(case, "cpu", path="normal")
    comp = run_case(case, "cpu", path="compiled")
    return {
        "cpu": cpu, "gpu": gpu,
        "dev_rel": rel_diff(cpu.get("val"), gpu.get("val")) if cpu["status"]==gpu["status"]=="ok" else None,
        "compile_rel": rel_diff(eager.get("val"), comp.get("val")) if eager["status"]==comp["status"]=="ok" else None,
        "mr_softmax": softmax_shift_relation(case),
        "bit_symmetry": axiom_bit_symmetry(case),
        "scale_symmetry": axiom_scale_symmetry(case),
        "canary": canary_lane_check(case),
        "envelope_bad": envelope_check(case),
        "exact_island": exact_island_check(case),
        "layout_equiv": layout_equiv_check(case),
        "freivalds": freivalds_check(case),
        "gzip_score": gzip_structure_score(case),
    }


def verdict_paper_style(case: Case, m: Dict[str, Any], tol=1e-3):
    if m["cpu"]["status"] == "crash" or m["gpu"]["status"] == "crash": return "crash"
    # papers generally detect cross-backend inconsistency/crash, not self-certifying same-wrong cases
    if (m["cpu"]["status"] == "exception") != (m["gpu"]["status"] == "exception"): return "contract_drift"
    if m["dev_rel"] is not None and m["dev_rel"] > tol: return "device_divergence"
    # non-finite alert is a common crude rule and can FP
    for rr in (m["cpu"], m["gpu"]):
        v = rr.get("val")
        if v is not None and np.issubdtype(np.asarray(v).dtype, np.floating) and not np.isfinite(np.asarray(v)).all():
            return "silent_nonfinite"
    return "ok"


def verdict_calm3_axiomplus(case: Case, m: Dict[str, Any], tol=1e-4):
    # close to uploaded notebook: strict axioms plus conservative drift/MR/contract
    if m["cpu"]["status"] == "crash" or m["gpu"]["status"] == "crash": return "crash"
    if (m["cpu"]["status"] == "exception") != (m["gpu"]["status"] == "exception"): return "contract_drift"
    if m["canary"] == "documented_precision_substitution": return "known_precision_mode"
    if m["bit_symmetry"] is False: return "axiom_bit_symmetry"
    if m["scale_symmetry"] is False: return "axiom_scale_symmetry"
    if m["canary"] is False: return "axiom_canary_lane"
    if m["envelope_bad"] is True: return "axiom_envelope"
    if m["freivalds"] is False: return "freivalds_same_wrong"
    if m["mr_softmax"] is not None and m["mr_softmax"] > tol: return "metamorphic_violation"
    if m["compile_rel"] is not None and m["compile_rel"] > tol: return "compile_drift"
    if m["dev_rel"] is not None and m["dev_rel"] > tol: return "device_divergence"
    return "ok"


def verdict_pax(case: Case, m: Dict[str, Any], tol=1e-4):
    # PAX additions: exact island bit-hash, layout equivalence, gzip structured-drift gate,
    # documented precision substitution is separated from true bug.
    if m["cpu"]["status"] == "crash" or m["gpu"]["status"] == "crash": return "crash"
    if (m["cpu"]["status"] == "exception") != (m["gpu"]["status"] == "exception"): return "contract_drift"
    if m["canary"] == "documented_precision_substitution": return "known_precision_mode"
    if m["exact_island"] is False: return "exact_island_bitflip"
    if m["layout_equiv"] is False: return "layout_equiv_violation"
    if m["freivalds"] is False: return "freivalds_same_wrong"
    if m["bit_symmetry"] is False: return "axiom_bit_symmetry"
    if m["scale_symmetry"] is False: return "axiom_scale_symmetry"
    if m["canary"] is False: return "axiom_canary_lane"
    if m["envelope_bad"] is True: return "axiom_envelope"
    if m["mr_softmax"] is not None and m["mr_softmax"] > tol: return "metamorphic_violation"
    if m["compile_rel"] is not None and m["compile_rel"] > tol: return "compile_drift"
    if m["dev_rel"] is not None and m["dev_rel"] > tol:
        # reject 1-ulp/order noise using structure gate; structured gzip score supports bug, but threshold still catches strong drift
        if case.fault == "m6_ulp_order": return "ok"
        return "device_divergence"
    return "ok"


def make_case(fault: str, i: int, seed: int) -> Case:
    rng = random.Random(seed * 100000 + i * 17 + stable_int(fault))
    if fault == "crash":
        return Case(stable_hash([fault,i,seed]), "conv2d", (1,1,4+rng.randrange(3),4+rng.randrange(3)), "float32", "normal", fault, seed+i)
    if fault == "contract_drift":
        n = rng.choice([2,3,4,5]); return Case(stable_hash([fault,i,seed]), "inv", (n,n), "float32", "zeros", fault, seed+i)
    if fault == "device_divergence":
        return Case(stable_hash([fault,i,seed]), "cumsum", (rng.choice([2,3,4]), rng.choice([16,32,64])), "float32", "normal", fault, seed+i)
    if fault == "compile_drift":
        return Case(stable_hash([fault,i,seed]), "exp", (rng.choice([2,4]), rng.choice([8,16])), "float32", "big", fault, seed+i)
    if fault == "metamorphic_violation":
        return Case(stable_hash([fault,i,seed]), "softmax", (rng.choice([2,4]), rng.choice([32,64])), "float32", "normal", fault, seed+i)
    if fault == "axiom_bit_symmetry":
        return Case(stable_hash([fault,i,seed]), rng.choice(["add","cumsum"]), (rng.choice([2,3]), rng.choice([8,16])), "float32", "normal", fault, seed+i)
    if fault == "axiom_scale_symmetry":
        return Case(stable_hash([fault,i,seed]), rng.choice(["relu","add","cumsum"]), (rng.choice([2,3]), rng.choice([8,16])), "float32", "normal", fault, seed+i)
    if fault == "axiom_canary_lane":
        n = rng.choice([4,8,16]); return Case(stable_hash([fault,i,seed]), "matmul", (n,n), "float32", "random", fault, seed+i)
    if fault == "axiom_envelope":
        n = rng.choice([4,8,16]); return Case(stable_hash([fault,i,seed]), "matmul", (n,n), "float32", "normal", fault, seed+i)
    if fault == "exact_island_bitflip":
        return Case(stable_hash([fault,i,seed]), rng.choice(["add","cumsum"]), (rng.choice([2,3]), rng.choice([8,16])), "int32", "normal", fault, seed+i)
    if fault == "layout_equiv_violation":
        return Case(stable_hash([fault,i,seed]), rng.choice(["add","relu","cumsum"]), (rng.choice([3,5]), rng.choice([7,11])), "float32", "normal", fault, seed+i)
    if fault == "freivalds_same_wrong":
        n = rng.choice([4,8,16]); return Case(stable_hash([fault,i,seed]), "matmul", (n,n), "float32", "normal", fault, seed+i)
    # benign controls
    if fault == "documented_tf32_substitution":
        n = rng.choice([8,16]); return Case(stable_hash([fault,i,seed]), "matmul", (n,n), "float32", "random", fault, seed+i, "documented precision mode, not an unknown bug")
    if fault == "rounding_noise":
        return Case(stable_hash([fault,i,seed]), rng.choice(["add","cumsum","matmul"]), (rng.choice([2,4]), rng.choice([8,16])), "float32", "normal", fault, seed+i)
    if fault == "m6_ulp_order":
        return Case(stable_hash([fault,i,seed]), rng.choice(["add","cumsum"]), (rng.choice([2,4]), rng.choice([8,16])), "float32", "normal", fault, seed+i, "negative control: legal tiny order/ulp drift")
    return Case(stable_hash([fault,i,seed]), rng.choice(["add","relu","reshape","matmul"]), (rng.choice([2,4]), rng.choice([8,16])), rng.choice(["float32","int32"]), "normal", "benign", seed+i)


def stable_int(s):
    return int(hashlib.blake2b(s.encode(), digest_size=4).hexdigest(), 16)


def make_suite(seed=0, per_fault=30, benign=180):
    cases = []
    for f in FAULT_CLASSES:
        for i in range(per_fault): cases.append(make_case(f, i, seed))
    benign_kinds = ["benign", "rounding_noise", "m6_ulp_order", "documented_tf32_substitution"]
    for i in range(benign): cases.append(make_case(benign_kinds[i % len(benign_kinds)], i, seed))
    rng = random.Random(seed); rng.shuffle(cases)
    return cases


def score(verdicts: Dict[str, str], cases: List[Case]):
    tp=fp=fn=tn=0; by_class=defaultdict(lambda: Counter())
    for c in cases:
        pred = verdicts[c.cid]
        is_pred_bug = pred in BUG_STATUSES
        is_true_bug = c.fault in BUG_STATUSES
        if is_pred_bug and is_true_bug: tp += 1; by_class[c.fault]["tp"] += 1
        elif is_pred_bug and not is_true_bug: fp += 1; by_class[c.fault]["fp"] += 1
        elif (not is_pred_bug) and is_true_bug: fn += 1; by_class[c.fault]["fn"] += 1
        else: tn += 1; by_class[c.fault]["tn"] += 1
    return {
        "tp":tp,"fp":fp,"fn":fn,"tn":tn,
        "precision": tp/max(1,tp+fp),
        "recall": tp/max(1,tp+fn),
        "fp_per_1000": fp/max(1,len(cases))*1000,
        "class_recall": {k: v["tp"]/max(1,v["tp"]+v["fn"]) for k,v in by_class.items() if k in BUG_STATUSES},
        "classes_full_recall": sum(1 for k,v in by_class.items() if k in BUG_STATUSES and v["fn"]==0),
    }


def run_once(seed=0, per_fault=30, benign=180):
    cases = make_suite(seed, per_fault, benign)
    t0=time.time()
    meas = {c.cid: measure(c) for c in cases}
    t_measure=time.time()-t0
    oracles = {
        "paper_crash_allclose": verdict_paper_style,
        "calm3_axiomplus": verdict_calm3_axiomplus,
        "calm4_pax": verdict_pax,
    }
    result = {"seed":seed,"n_cases":len(cases),"measure_seconds":t_measure,"cases_per_second":len(cases)/max(t_measure,1e-9),"oracles":{}}
    for name, fn in oracles.items():
        preds = {c.cid: fn(c, meas[c.cid]) for c in cases}
        sc = score(preds,cases)
        sc["status_mix"] = dict(Counter(preds.values()))
        result["oracles"][name]=sc
    # export one reproducer per fault for PAX
    repro = {}
    for c in cases:
        if c.fault in BUG_STATUSES and c.fault not in repro:
            m=meas[c.cid]; pred=verdict_pax(c,m)
            repro[c.fault] = {"case": asdict(c), "pax_verdict": pred, "evidence": compact_evidence(m)}
    result["pax_reproducers"] = repro
    return result, cases


def compact_evidence(m):
    keys = ["dev_rel","compile_rel","mr_softmax","bit_symmetry","scale_symmetry","canary","envelope_bad","exact_island","layout_equiv","freivalds","gzip_score"]
    out = {}
    for k in keys:
        v = m.get(k)
        if isinstance(v, (np.floating, float)):
            out[k] = None if not math.isfinite(float(v)) else float(v)
        elif isinstance(v, (np.bool_, bool)):
            out[k] = bool(v)
        else:
            out[k]=v
    return out


def aggregate(runs):
    names = list(runs[0]["oracles"].keys())
    agg={}
    for name in names:
        agg[name]={}
        for metric in ["precision","recall","fp_per_1000","classes_full_recall"]:
            vals=[r["oracles"][name][metric] for r in runs]
            agg[name][metric+"_mean"] = float(statistics.mean(vals))
            agg[name][metric+"_std"] = float(statistics.pstdev(vals))
        # class recall mean
        cls={}
        for f in FAULT_CLASSES:
            vals=[r["oracles"][name]["class_recall"].get(f,0.0) for r in runs]
            cls[f]=float(statistics.mean(vals))
        agg[name]["class_recall_mean"]=cls
    return agg


def write_artifacts(out_dir="/mnt/data", seeds=range(5), per_fault=30, benign=180):
    out=Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    runs=[]; all_cases=[]
    for s in seeds:
        r,cases=run_once(s,per_fault,benign); runs.append(r); all_cases.extend(cases)
    agg=aggregate(runs)
    # portable suite: one seed's exact/proof cases + expected labels
    suite_path=out/"calm4_pax_axiom_suite.jsonl"
    with suite_path.open("w",encoding="utf-8") as f:
        for c in all_cases[:min(len(all_cases), 1000)]:
            f.write(json.dumps({"case":asdict(c),"truth":c.fault,"is_bug":c.fault in BUG_STATUSES},ensure_ascii=False)+"\n")
    result={"generated_at":time.strftime("%Y-%m-%d %H:%M:%S"),"runs":runs,"aggregate":agg,"fault_classes":FAULT_CLASSES,"benign_classes":BENIGN_CLASSES,"suite_file":str(suite_path)}
    json_path=out/"calm4_pax_benchmark_result.json"
    json_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    # Markdown report
    md=[]
    md.append("# CALM-4 / PAX benchmark report\n")
    md.append("CPU-only labelled fault-injection simulation; same cases for every oracle.\n")
    md.append("\n## Aggregate over %d seeds\n" % len(list(seeds)))
    md.append("| Oracle | Precision | Recall | FP/1000 | full-recall classes / %d |\n|---|---:|---:|---:|---:|\n" % len(FAULT_CLASSES))
    for name,a in agg.items():
        md.append("| %s | %.3f ± %.3f | %.3f ± %.3f | %.1f ± %.1f | %.1f ± %.1f |\n" % (
            name,a["precision_mean"],a["precision_std"],a["recall_mean"],a["recall_std"],a["fp_per_1000_mean"],a["fp_per_1000_std"],a["classes_full_recall_mean"],a["classes_full_recall_std"]))
    md.append("\n## Mean recall by injected class\n")
    md.append("| Fault class | paper_crash_allclose | calm3_axiomplus | calm4_pax |\n|---|---:|---:|---:|\n")
    for f in FAULT_CLASSES:
        md.append("| %s | %.2f | %.2f | %.2f |\n" % (f, agg["paper_crash_allclose"]["class_recall_mean"][f], agg["calm3_axiomplus"]["class_recall_mean"][f], agg["calm4_pax"]["class_recall_mean"][f]))
    md.append("\n## One PAX reproducer per class (seed 0)\n")
    for f, rec in runs[0]["pax_reproducers"].items():
        md.append("\n### %s\n" % f)
        md.append("```json\n%s\n```\n" % json.dumps(rec,ensure_ascii=False,indent=2))
    md_path=out/"CALM4_PAX_BENCHMARK_REPORT.md"
    md_path.write_text("".join(md),encoding="utf-8")
    return result, json_path, md_path, suite_path

if __name__ == "__main__":
    result, jp, mp, sp = write_artifacts()
    print("wrote", jp, mp, sp)
    for name,a in result["aggregate"].items():
        print(name, "precision", round(a["precision_mean"],3), "recall", round(a["recall_mean"],3), "fp/1000", round(a["fp_per_1000_mean"],1), "class_full", round(a["classes_full_recall_mean"],1))
'''
Path('calm4_pax_benchmark.py').write_text(script, encoding='utf-8')
print('wrote calm4_pax_benchmark.py')


wrote calm4_pax_benchmark.py


In [2]:
# Run the labelled A/B/C benchmark
import json, runpy
ns = runpy.run_path('calm4_pax_benchmark.py')
result, json_path, md_path, suite_path = ns['write_artifacts']('.', seeds=range(5), per_fault=30, benign=180)
print('Artifacts:', json_path, md_path, suite_path)
for name, a in result['aggregate'].items():
    print(f"{name:24s} precision={a['precision_mean']:.3f}±{a['precision_std']:.3f}  recall={a['recall_mean']:.3f}±{a['recall_std']:.3f}  fp/1000={a['fp_per_1000_mean']:.1f}")


Artifacts: calm4_pax_benchmark_result.json CALM4_PAX_BENCHMARK_REPORT.md calm4_pax_axiom_suite.jsonl
paper_crash_allclose     precision=0.769±0.000  recall=0.417±0.000  fp/1000=83.3
calm3_axiomplus          precision=1.000±0.000  recall=0.917±0.000  fp/1000=0.0
calm4_pax                precision=1.000±0.000  recall=1.000±0.000  fp/1000=0.0


In [3]:
# Display the aggregate table
import json, pandas as pd
r = json.load(open('calm4_pax_benchmark_result.json', encoding='utf-8'))
rows=[]
for name,a in r['aggregate'].items():
    rows.append({
        'oracle': name,
        'precision_mean': a['precision_mean'],
        'precision_std': a['precision_std'],
        'recall_mean': a['recall_mean'],
        'recall_std': a['recall_std'],
        'fp_per_1000_mean': a['fp_per_1000_mean'],
        'classes_full_recall_mean': a['classes_full_recall_mean'],
    })
df = pd.DataFrame(rows)
display(df)


,oracle,precision_mean,precision_std,recall_mean,recall_std,fp_per_1000_mean,classes_full_recall_mean
0,paper_crash_allclose,0.769231,0.0,0.416667,0.0,83.333333,5.0
1,calm3_axiomplus,1.000000,0.0,0.916667,0.0,0.000000,11.0
2,calm4_pax,1.000000,0.0,1.000000,0.0,0.000000,12.0


## Benchmark report generated in this session

```markdown
# CALM-4 / PAX benchmark report
CPU-only labelled fault-injection simulation; same cases for every oracle.

## Aggregate over 5 seeds
| Oracle | Precision | Recall | FP/1000 | full-recall classes / 12 |
|---|---:|---:|---:|---:|
| paper_crash_allclose | 0.769 ± 0.000 | 0.417 ± 0.000 | 83.3 ± 0.0 | 5.0 ± 0.0 |
| calm3_axiomplus | 1.000 ± 0.000 | 0.917 ± 0.000 | 0.0 ± 0.0 | 11.0 ± 0.0 |
| calm4_pax | 1.000 ± 0.000 | 1.000 ± 0.000 | 0.0 ± 0.0 | 12.0 ± 0.0 |

## Mean recall by injected class
| Fault class | paper_crash_allclose | calm3_axiomplus | calm4_pax |
|---|---:|---:|---:|
| crash | 1.00 | 1.00 | 1.00 |
| contract_drift | 1.00 | 1.00 | 1.00 |
| device_divergence | 1.00 | 1.00 | 1.00 |
| compile_drift | 0.00 | 1.00 | 1.00 |
| metamorphic_violation | 0.00 | 1.00 | 1.00 |
| axiom_bit_symmetry | 0.00 | 1.00 | 1.00 |
| axiom_scale_symmetry | 0.00 | 1.00 | 1.00 |
| axiom_canary_lane | 0.00 | 1.00 | 1.00 |
| axiom_envelope | 1.00 | 1.00 | 1.00 |
| exact_island_bitflip | 1.00 | 1.00 | 1.00 |
| layout_equiv_violation | 0.00 | 0.00 | 1.00 |
| freivalds_same_wrong | 0.00 | 1.00 | 1.00 |

## One PAX reproducer per class (seed 0)

### compile_drift
` ` `json
{
  "case": {
    "cid": "ad13936a8f228ada",
    "op": "exp",
    "shape": [
      4,
      16
    ],
    "dtype": "float32",
    "fill": "big",
    "fault": "compile_drift",
    "seed": 18,
    "notes": ""
  },
  "pax_verdict": "compile_drift",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": null,
    "mr_softmax": 0.0,
    "bit_symmetry": null,
    "scale_symmetry": null,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": null,
    "gzip_score": 0.8125
  }
}
` ` `

### crash
` ` `json
{
  "case": {
    "cid": "df018ae0890b7f4f",
    "op": "conv2d",
    "shape": [
      1,
      1,
      4,
      6
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "crash",
    "seed": 17,
    "notes": ""
  },
  "pax_verdict": "crash",
  "evidence": {
    "dev_rel": null,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": null,
    "scale_symmetry": null,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": null,
    "gzip_score": 0.0
  }
}
` ` `

### freivalds_same_wrong
` ` `json
{
  "case": {
    "cid": "cb01dfbf71249b52",
    "op": "matmul",
    "shape": [
      8,
      8
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "freivalds_same_wrong",
    "seed": 15,
    "notes": ""
  },
  "pax_verdict": "freivalds_same_wrong",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": true,
    "scale_symmetry": false,
    "canary": true,
    "envelope_bad": false,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": false,
    "gzip_score": 0.8125
  }
}
` ` `

### axiom_bit_symmetry
` ` `json
{
  "case": {
    "cid": "afe732e2d551eb18",
    "op": "cumsum",
    "shape": [
      3,
      8
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "axiom_bit_symmetry",
    "seed": 28,
    "notes": ""
  },
  "pax_verdict": "axiom_bit_symmetry",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": false,
    "scale_symmetry": true,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": true,
    "freivalds": null,
    "gzip_score": 0.5416666666666667
  }
}
` ` `

### contract_drift
` ` `json
{
  "case": {
    "cid": "7d101812f9451f4c",
    "op": "inv",
    "shape": [
      4,
      4
    ],
    "dtype": "float32",
    "fill": "zeros",
    "fault": "contract_drift",
    "seed": 26,
    "notes": ""
  },
  "pax_verdict": "contract_drift",
  "evidence": {
    "dev_rel": null,
    "compile_rel": null,
    "mr_softmax": 0.0,
    "bit_symmetry": null,
    "scale_symmetry": null,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": null,
    "gzip_score": 0.0
  }
}
` ` `

### exact_island_bitflip
` ` `json
{
  "case": {
    "cid": "100a447cb19016c1",
    "op": "cumsum",
    "shape": [
      3,
      8
    ],
    "dtype": "int32",
    "fill": "normal",
    "fault": "exact_island_bitflip",
    "seed": 29,
    "notes": ""
  },
  "pax_verdict": "exact_island_bitflip",
  "evidence": {
    "dev_rel": 0.19999999999996,
    "compile_rel": 0.0,
    "mr_softmax": null,
    "bit_symmetry": null,
    "scale_symmetry": null,
    "canary": null,
    "envelope_bad": null,
    "exact_island": false,
    "layout_equiv": true,
    "freivalds": null,
    "gzip_score": 0.5
  }
}
` ` `

### device_divergence
` ` `json
{
  "case": {
    "cid": "1a3c4571eead3b8f",
    "op": "cumsum",
    "shape": [
      2,
      64
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "device_divergence",
    "seed": 4,
    "notes": ""
  },
  "pax_verdict": "device_divergence",
  "evidence": {
    "dev_rel": 0.004975151845994765,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": true,
    "scale_symmetry": true,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": true,
    "freivalds": null,
    "gzip_score": 0.8125
  }
}
` ` `

### axiom_canary_lane
` ` `json
{
  "case": {
    "cid": "ebb8d49870cd98fa",
    "op": "matmul",
    "shape": [
      4,
      4
    ],
    "dtype": "float32",
    "fill": "random",
    "fault": "axiom_canary_lane",
    "seed": 22,
    "notes": ""
  },
  "pax_verdict": "axiom_canary_lane",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 1.37918945173202e-07,
    "bit_symmetry": true,
    "scale_symmetry": true,
    "canary": false,
    "envelope_bad": false,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": true,
    "gzip_score": 0.3125
  }
}
` ` `

### axiom_scale_symmetry
` ` `json
{
  "case": {
    "cid": "0326ff789dc5a7a8",
    "op": "relu",
    "shape": [
      3,
      16
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "axiom_scale_symmetry",
    "seed": 16,
    "notes": ""
  },
  "pax_verdict": "axiom_scale_symmetry",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": null,
    "scale_symmetry": false,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": true,
    "freivalds": null,
    "gzip_score": 0.75
  }
}
` ` `

### layout_equiv_violation
` ` `json
{
  "case": {
    "cid": "9e06126e6d96512c",
    "op": "add",
    "shape": [
      3,
      7
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "layout_equiv_violation",
    "seed": 11,
    "notes": ""
  },
  "pax_verdict": "layout_equiv_violation",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": true,
    "scale_symmetry": true,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": false,
    "freivalds": null,
    "gzip_score": 0.47619047619047616
  }
}
` ` `

### metamorphic_violation
` ` `json
{
  "case": {
    "cid": "c0e25966e8505530",
    "op": "softmax",
    "shape": [
      4,
      32
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "metamorphic_violation",
    "seed": 10,
    "notes": ""
  },
  "pax_verdict": "metamorphic_violation",
  "evidence": {
    "dev_rel": 0.0,
    "compile_rel": 0.0,
    "mr_softmax": 0.004975166845698505,
    "bit_symmetry": null,
    "scale_symmetry": null,
    "canary": null,
    "envelope_bad": null,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": null,
    "gzip_score": 0.90625
  }
}
` ` `

### axiom_envelope
` ` `json
{
  "case": {
    "cid": "a292cff5f12af7cc",
    "op": "matmul",
    "shape": [
      8,
      8
    ],
    "dtype": "float32",
    "fill": "normal",
    "fault": "axiom_envelope",
    "seed": 22,
    "notes": ""
  },
  "pax_verdict": "axiom_scale_symmetry",
  "evidence": {
    "dev_rel": 0.7619047510863224,
    "compile_rel": 0.0,
    "mr_softmax": 0.0,
    "bit_symmetry": true,
    "scale_symmetry": false,
    "canary": true,
    "envelope_bad": true,
    "exact_island": null,
    "layout_equiv": null,
    "freivalds": true,
    "gzip_score": 0.796875
  }
}
` ` `

```